In [1]:
"""!pip install --upgrade pylace
"""

'!pip install --upgrade pylace\n'

In [2]:
# lace.ipynb – Cell 1: Setup

import pandas as pd
pd.set_option("display.notebook_repr_html", False)

import numpy as np

import lace
from lace.plot import diagnostics  # for convergence plots

print("lace version:", lace.__version__)


lace version: 0.9.0


In [3]:
# lace.ipynb – Cell 2: Load cleaned data for Lace

import os

print("Files in CWD:", os.listdir())

df = pd.read_csv("cleaned_data.csv")  # you saved index=False, so no index_col here
print("Raw shape:", df.shape)
df.head()


Files in CWD: ['.git', '.gitignore', 'cleaned_data.csv', 'data cleaning.ipynb', 'lace copy.ipynb', 'lace.ipynb', 'Lateral Data 2014-2022.xlsx', 'LICENSE', 'README.md', 'try copy.ipynb', 'try.ipynb', 'uncleaned_data.csv']
Raw shape: (546, 115)


  Date of Surgery  Sex  Age       Case/Type of Surgery  Perc screws?  Open  \
0       1/28/2014    0   63  llif l4-l5 w/ perc screws             1     0   
1       1/28/2014    0   55                 llif l2-l5             0     0   
2       1/29/2014    1   74                 llif l2-l3             0     0   
3       2/10/2014    1   55                llif l3-l5              0     0   
4       2/24/2014    1   67                 llif l4-l5             0     0   

   Open Check V2  Standalone XLIF Check  \
0              0                      0   
1              0                      1   
2              0                      1   
3              0                      1   
4              0                      1   

    Retroperitoneal Approach (LLIF ± ALIF)  Anterior + Posterior Apporoach  \
0                                        0                               1   
1                                        1                               0   
2                                     

In [4]:
# lace.ipynb – Cell 3: Light cleanup for Lace

df_lace = df.copy()

drop_cols = []

# 1) Validation column if present
if "VALIDATION COLUMN" in df_lace.columns:
    drop_cols.append("VALIDATION COLUMN")

# 2) Duplicate complication columns with ".1" suffix
drop_cols.extend([c for c in df_lace.columns if c.endswith(".1")])

# 3) Date-like columns (keep the derived durations instead)
date_like = [
    "Date of Surgery",
    "post-op date",
    "most recent date",
    "discharge date",
    "Last follow-up date",
]
drop_cols.extend([c for c in date_like if c in df_lace.columns])

# Make unique
drop_cols = list(dict.fromkeys(drop_cols))

print("Dropping columns:", drop_cols)
df_lace = df_lace.drop(columns=drop_cols)
print("Shape after Lace cleanup:", df_lace.shape)

df_lace.head()


Dropping columns: ['VALIDATION COLUMN', 'acute thigh paresthesia (immediate post op).1', 'transient paresthesia (post op clinic note).1', 'psoas hematoma.1', 'abdominal hernia (post op clinic note).1', 'Date of Surgery', 'post-op date', 'most recent date', 'discharge date', 'Last follow-up date']
Shape after Lace cleanup: (546, 105)


   Sex  Age       Case/Type of Surgery  Perc screws?  Open  Open Check V2  \
0    0   63  llif l4-l5 w/ perc screws             1     0              0   
1    0   55                 llif l2-l5             0     0              0   
2    1   74                 llif l2-l3             0     0              0   
3    1   55                llif l3-l5              0     0              0   
4    1   67                 llif l4-l5             0     0              0   

   Standalone XLIF Check   Retroperitoneal Approach (LLIF ± ALIF)  \
0                      0                                        0   
1                      1                                        1   
2                      1                                        1   
3                      1                                        1   
4                      1                                        1   

   Anterior + Posterior Apporoach  Osteotomies (yes/no)  ... alif_text  \
0                               1               

In [5]:
# Start from df_lace again
import numpy as np

# 1) Drop columns with <= 1 unique *non-null* value
constant_nonnull_cols = [
    c for c in df_lace.columns
    if df_lace[c].dropna().nunique() <= 1
]

print("Dropping constant (non-null) columns:", constant_nonnull_cols)

df_lace_model = df_lace.drop(columns=constant_nonnull_cols)

print("Original shape:", df_lace.shape)
print("Modeling shape:", df_lace_model.shape)


Dropping constant (non-null) columns: ['T12-L1', 'L1-L2', 'L2-L3', 'L3-L4', 'L4-L5', 'L5-S1', 'Hernia', 'abdominal hernia (post op clinic note)', 'levels_fused_count', 'construct_span_levels', 'thoracolumbar_junction', 'upper_lumbar', 'lower_lumbar', 'lumbosacral']
Original shape: (546, 105)
Modeling shape: (546, 91)


In [6]:
from lace import Codebook

codebook = Codebook.from_df("spine_lateral_llif", df_lace_model)
codebook


Codebook 'spine_lateral_llif'
  state_prior_process: None
  view_prior_process: None
  columns: 91
  rows: 546

In [9]:
from lace import Engine

engine = Engine.from_df(df_lace_model, codebook=codebook)
engine.update(5_000)   # test run


0it [00:00, ?it/s]

In [10]:
from lace.plot import diagnostics
diagnostics(engine)


In [11]:
depmap = engine.clustermap("depprob", zmin=0, zmax=1)
depmap.figure.show()


In [12]:
# Choose clinically meaningful variables
target_cols = [
    "post op ODI",
    "post op VAS back (media tab)",
    "post op VAS leg (media tab)",
    "radiographic adjacent segment disease? (Y/N)",
    "need revision surgery? (Y=1)",
]

preop_cols = [
    "Age", "Sex", "BMI",
    "pre op ODI", "pre op VAS back", "pre op VAS leg",
    "Average PI", "PI-LL angle mismatch", "ABS PI-LL angle mismatch",
    "prior back surgeries? (y=1)",
    "dx_adjacent_segment", "dx_spondylolisthesis", "dx_stenosis", "dx_deformity",
]

surg_cols = [
    "LLIF?",
    "Perc screws?",
    "Open",
    "Standalone XLIF Check",
    "Retroperitoneal Approach (LLIF ± ALIF)",
    "Anterior + Posterior Apporoach",
    "ALIF Count",
    "Lateral Count",
    "alif_text",
    "xlif_text",
    "revision_surgery",
]

# Filter columns that exist in df_lace_model
cols = [c for c in (target_cols + preop_cols + surg_cols) if c in df_lace_model.columns]

len(cols), cols


(28,
 ['post op ODI',
  'post op VAS back (media tab)',
  'post op VAS leg (media tab)',
  'radiographic adjacent segment disease? (Y/N)',
  'need revision surgery? (Y=1)',
  'Age',
  'Sex',
  'BMI',
  'pre op ODI',
  'pre op VAS back',
  'pre op VAS leg',
  'Average PI',
  'PI-LL angle mismatch',
  'ABS PI-LL angle mismatch',
  'prior back surgeries? (y=1)',
  'dx_adjacent_segment',
  'dx_spondylolisthesis',
  'dx_stenosis',
  'dx_deformity',
  'Perc screws?',
  'Open',
  'Standalone XLIF Check',
  'Anterior + Posterior Apporoach',
  'ALIF Count',
  'Lateral Count',
  'alif_text',
  'xlif_text',
  'revision_surgery'])

In [13]:
depmap_small = engine.clustermap(
    "depprob",
    indices=cols,
    zmin=0,
    zmax=1
)
depmap_small.figure.show()


In [17]:
import pandas as pd

dp_all = engine.pairwise_fn("depprob", indices=cols)  # polars DataFrame
dp_all_pd = dp_all.to_pandas()
dp_all_pd.head()

target = "post op ODI"

dp_odi = dp_all_pd[
    (dp_all_pd["A"] == target) | (dp_all_pd["B"] == target)
].copy()

# Get the "other" variable in each pair
dp_odi["other"] = dp_odi.apply(
    lambda row: row["B"] if row["A"] == target else row["A"],
    axis=1
)

# Drop the trivial self-pair if present
dp_odi = dp_odi[dp_odi["other"] != target]

# Sort by depprob
dp_odi_sorted = dp_odi.sort_values("depprob", ascending=False)

dp_odi_sorted.head(15)


                                A                             B  depprob  \
1                     post op ODI  post op VAS back (media tab)    1.000   
9                     post op ODI               pre op VAS back    1.000   
280                pre op VAS leg                   post op ODI    1.000   
252               pre op VAS back                   post op ODI    1.000   
224                    pre op ODI                   post op ODI    1.000   
56    post op VAS leg (media tab)                   post op ODI    1.000   
2                     post op ODI   post op VAS leg (media tab)    1.000   
10                    post op ODI                pre op VAS leg    1.000   
28   post op VAS back (media tab)                   post op ODI    1.000   
8                     post op ODI                    pre op ODI    1.000   
476                   dx_stenosis                   post op ODI    0.375   
17                    post op ODI                   dx_stenosis    0.375   
26          

In [18]:
for outcome in [
    "radiographic adjacent segment disease? (Y/N)",
    "need revision surgery? (Y=1)"
]:
    if outcome in cols:
        dp_out = dp_all_pd[
            (dp_all_pd["A"] == outcome) | (dp_all_pd["B"] == outcome)
        ].copy()
        dp_out["other"] = dp_out.apply(
            lambda row: row["B"] if row["A"] == outcome else row["A"],
            axis=1
        )
        dp_out = dp_out[dp_out["other"] != outcome]
        print("\n=== Dependence with", outcome, "===")
        print(dp_out.sort_values("depprob", ascending=False).head(10))



=== Dependence with radiographic adjacent segment disease? (Y/N) ===
                                                A  \
115                  need revision surgery? (Y=1)   
88   radiographic adjacent segment disease? (Y/N)   
647                                    ALIF Count   
107  radiographic adjacent segment disease? (Y/N)   
109  radiographic adjacent segment disease? (Y/N)   
703                                     alif_text   
90   radiographic adjacent segment disease? (Y/N)   
171                                           Sex   
675                                 Lateral Count   
339                          PI-LL angle mismatch   

                                                B  depprob  \
115  radiographic adjacent segment disease? (Y/N)    0.375   
88                   need revision surgery? (Y=1)    0.375   
647  radiographic adjacent segment disease? (Y/N)    0.250   
107                                    ALIF Count    0.250   
109                                 

In [19]:
# Most surprising ODI values
surp_odi = engine.surprisal("post op ODI").to_pandas()
surp_odi_sorted = surp_odi.sort_values("surprisal", ascending=False)
surp_odi_sorted.head(10)


    index post op ODI  surprisal
128   414           7   3.798416
32     91          76   3.773165
57    182          28   3.714805
89    277     missing   3.546021
29     88          44   3.467779
126   399          10   3.372502
35     99          34   3.371242
12     47          18   3.368694
106   299           2   3.258318
69    250          22   3.242367

In [23]:
# Grab the indices of the top-10 surprising ODI rows
top_ids = surp_odi_sorted["index"].head(10).astype(int).tolist()
df_lace_model.loc[top_ids, [
    "Age", "Sex", "BMI",
    "pre op ODI", "pre op VAS back", "pre op VAS leg",
    "post op ODI", "post op VAS back (media tab)", "post op VAS leg (media tab)",
    "dx_stenosis", "dx_deformity", "dx_spondylolisthesis",
    "ALIF Count", "Lateral Count", "alif_text", "xlif_text",
    "radiographic adjacent segment disease? (Y/N)",
    "need revision surgery? (Y=1)"
]]


     Age  Sex    BMI pre op ODI pre op VAS back pre op VAS leg post op ODI  \
ID                                                                           
414   60    0  33.00         44               8              9           7   
91    65    0  31.20         64               8              8          76   
182   76    0  31.00         56               9              8          28   
277   72    1  36.44    Missing         Missing        Missing     missing   
88    57    1  33.40    Missing         Missing        Missing          44   
399   50    1  34.00         48               9              9          10   
99    67    0  30.55         60               8              8          34   
47    67    0  22.00         44               8              9          18   
299   53    1  34.70         62              10              2           2   
250   59    1  37.97         48               8              8          22   

    post op VAS back (media tab) post op VAS leg (media tab)  d

In [24]:
df_ml = df_lace_model.copy()

# require both pre and post ODI present
mask = df_ml["pre op ODI"].notna() & df_ml["post op ODI"].notna()
df_ml = df_ml[mask].copy()

df_ml["delta_odi"] = df_ml["pre op ODI"] - df_ml["post op ODI"]
df_ml["odi_responder"] = (df_ml["delta_odi"] >= 15).astype(int)  # example MCID
df_ml["odi_responder"].value_counts()


TypeError: unsupported operand type(s) for -: 'str' and 'str'

In [20]:
if "need revision surgery? (Y=1)" in df_lace_model.columns:
    surp_rev = engine.surprisal("need revision surgery? (Y=1)").to_pandas()
    surp_rev_sorted = surp_rev.sort_values("surprisal", ascending=False)
    surp_rev_sorted.head(10)
